# CBIR Pipeline - Preprocessing Method 3

This notebook trains Method 3 using only `data/train`.

Output artifacts are saved to `index/cbir_method3_index.npz` and `index/cbir_method3_meta.json`.

In [ ]:
from __future__ import annotations
from mediapipe.tasks.python import vision as mp_vision
from pathlib import Path
from typing import Any
import json

import cv2
import numpy as np
from numpy.typing import NDArray
import face_recognition
import mediapipe as mp
import urllib.request
ImageArray = NDArray[np.uint8]
FloatArray = NDArray[np.float32]

project_root: Path = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent

TRAIN_DIR: Path = project_root / "data" / "face"
INDEX_DIR: Path = project_root / "index"
INDEX_DIR.mkdir(parents=True, exist_ok=True)

CBIR_INDEX_PATH: Path = INDEX_DIR / "cbir_method3_index.npz"
CBIR_META_PATH: Path = INDEX_DIR / "cbir_method3_meta.json"
CBIR_SPLIT_PATH: Path = INDEX_DIR / "cbir_eval_split.json"

OUTPUT_SIZE: tuple[int, int] = (128, 128)
IMAGE_EXTS: set[str] = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print(f"Train raw folder        : {TRAIN_DIR}")
print(f"CBIR method3 index output: {CBIR_INDEX_PATH}")
print(f"CBIR method3 meta output : {CBIR_META_PATH}")
print(f"Shared split path        : {CBIR_SPLIT_PATH}")

Train raw folder        : c:\Users\jaft9\School\AttSystem\data\face
CBIR method3 index output: c:\Users\jaft9\School\AttSystem\index\cbir_method3_index.npz
CBIR method3 meta output : c:\Users\jaft9\School\AttSystem\index\cbir_method3_meta.json
Shared split path        : c:\Users\jaft9\School\AttSystem\index\cbir_eval_split.json


In [ ]:
_FACE_DETECTOR = None

def get_face_detector():
    global _FACE_DETECTOR
    if _FACE_DETECTOR is not None:
        return _FACE_DETECTOR
        

    
    model_path = project_root / "models" / "blaze_face_short_range.tflite"
    if not model_path.exists():
        model_path.parent.mkdir(parents=True, exist_ok=True)
        print("Downloading BlazeFace model...")
        url = "https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite"
        urllib.request.urlretrieve(url, str(model_path))
        
    options = mp_vision.FaceDetectorOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=str(model_path)),
        min_detection_confidence=0.5
    )
    _FACE_DETECTOR = mp_vision.FaceDetector.create_from_options(options)
    return _FACE_DETECTOR


def detect_face_roi(gray: ImageArray) -> ImageArray:
    import mediapipe as mp
    face_detector = get_face_detector()
    
    h, w = gray.shape[:2]
    rgb = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    results = face_detector.detect(mp_image)
    
    if not results.detections:
        side: int = int(min(h, w) * 0.75)
        cx, cy = w // 2, h // 2
        x1: int = max(0, cx - side // 2)
        y1: int = max(0, cy - side // 2)
        x2: int = min(w, x1 + side)
        y2: int = min(h, y1 + side)
        return gray[y1:y2, x1:x2].astype(np.uint8, copy=False)
        
    best_detection = max(
        results.detections,
        key=lambda d: d.bounding_box.width * d.bounding_box.height
    )
    bboxC = best_detection.bounding_box
    x = int(bboxC.origin_x)
    y = int(bboxC.origin_y)
    fw = int(bboxC.width)
    fh = int(bboxC.height)
    
    pad_w: int = int(fw * 0.2)
    pad_h: int = int(fh * 0.2)

    x1: int = max(0, x - pad_w)
    y1: int = max(0, y - pad_h)
    x2: int = min(w, x + fw + pad_w)
    y2: int = min(h, y + fh + pad_h)

    return gray[y1:y2, x1:x2].astype(np.uint8, copy=False)


def preprocess_roi_method3(roi_gray: ImageArray) -> ImageArray:
    equalized = cv2.equalizeHist(roi_gray)
    denoised = cv2.bilateralFilter(equalized, d=7, sigmaColor=50, sigmaSpace=50)
    sharpened = cv2.addWeighted(denoised, 1.35, cv2.GaussianBlur(denoised, (0, 0), 1.2), -0.35, 0)
    resized = cv2.resize(sharpened, OUTPUT_SIZE, interpolation=cv2.INTER_CUBIC)
    return resized.astype(np.uint8, copy=False)


def extract_embedding_from_face(face_gray: ImageArray) -> FloatArray | None:
    face_rgb = cv2.cvtColor(face_gray, cv2.COLOR_GRAY2RGB)
    h, w = face_rgb.shape[:2]
    # Force feature extraction on the already-cropped face region
    known_face_locations = [(0, w, h, 0)]
    encodings: list[Any] = face_recognition.face_encodings(face_rgb, known_face_locations=known_face_locations)
    if len(encodings) == 0:
        return None

    emb = np.array(encodings[0], dtype=np.float32)
    return emb


def build_cbir_gallery_method3(
    raw_dir: Path,
    split_data: dict[str, dict[str, list[str]]],
) -> tuple[FloatArray, NDArray[np.int32], list[str], list[str]]:
    if not raw_dir.exists() or not raw_dir.is_dir():
        raise FileNotFoundError(f"Missing dataset folder: {raw_dir}")

    person_dirs: list[Path] = sorted([p for p in raw_dir.iterdir() if p.is_dir()])
    if not person_dirs:
        raise RuntimeError("No person folders found in data/face.")

    embeddings: list[FloatArray] = []
    label_ids: list[int] = []
    label_names: list[str] = []
    image_paths: list[str] = []

    total_gallery_images: int = 0
    kept_images: int = 0

    for label_id, person_dir in enumerate(person_dirs):
        files: list[Path] = sorted([f for f in person_dir.iterdir() if f.is_file() and f.suffix.lower() in IMAGE_EXTS])
        if not files:
            continue

        selected_names = set(split_data.get(person_dir.name, {}).get("gallery", []))
        if selected_names:
            files = [f for f in files if f.name in selected_names]

        if not files:
            continue

        label_names.append(person_dir.name)
        person_kept: int = 0

        for f in files:
            total_gallery_images += 1
            bgr = cv2.imread(str(f), cv2.IMREAD_COLOR)
            if bgr is None:
                continue

            gray = cv2.cvtColor(bgr.astype(np.uint8, copy=False), cv2.COLOR_BGR2GRAY).astype(np.uint8, copy=False)
            roi = detect_face_roi(gray)
            if roi.size == 0:
                continue

            sample = preprocess_roi_method3(roi)
            emb = extract_embedding_from_face(sample)
            if emb is None:
                continue

            embeddings.append(emb)
            label_ids.append(label_id)
            image_paths.append(str(f))
            person_kept += 1
            kept_images += 1

        print(f"{person_dir.name}: indexed {person_kept}/{len(files)} gallery images")

    if len(embeddings) == 0:
        raise RuntimeError("No usable face embeddings were produced from gallery images.")

    print(f"Total train images read: {total_gallery_images}, indexed: {kept_images}")

    emb_np: FloatArray = np.vstack(embeddings).astype(np.float32)
    label_np: NDArray[np.int32] = np.array(label_ids, dtype=np.int32)
    return emb_np, label_np, label_names, image_paths

In [28]:
def _normalize_split_payload(split_payload: dict[str, Any]) -> dict[str, dict[str, list[str]]]:
    if "identities" in split_payload and isinstance(split_payload["identities"], dict):
        identities = split_payload["identities"]
    else:
        identities = split_payload

    normalized: dict[str, dict[str, list[str]]] = {}
    for person_name, payload in identities.items():
        if not isinstance(payload, dict):
            continue
        gallery = [str(x) for x in payload.get("gallery", [])]
        test = [str(x) for x in payload.get("test", [])]
        normalized[str(person_name)] = {"gallery": gallery, "test": test}
    return normalized

def save_cbir_method3_index() -> None:
    if not CBIR_SPLIT_PATH.exists():
        raise FileNotFoundError(f"Missing shared split data {CBIR_SPLIT_PATH}. Please run cbir_method1 first to generate the split.")

    with open(CBIR_SPLIT_PATH, "r", encoding="utf-8") as f:
        split_payload = json.load(f)
    split_data = _normalize_split_payload(split_payload)

    embeddings, labels, label_names, image_paths = build_cbir_gallery_method3(TRAIN_DIR, split_data)

    np.savez_compressed(
        CBIR_INDEX_PATH,
        embeddings=embeddings,
        labels=labels,
        image_paths=np.array(image_paths, dtype=object),
    )

    meta = {
        "embedding_dim": int(embeddings.shape[1]),
        "num_vectors": int(embeddings.shape[0]),
        "label_names": label_names,
        "distance": "euclidean",
        "source_folder": str(TRAIN_DIR),
        "split_path": str(CBIR_SPLIT_PATH),
        "preprocess_method": "method3",
    }

    with open(CBIR_META_PATH, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    print("\nCBIR Method 3 pipeline complete")
    print(f"Index saved: {CBIR_INDEX_PATH}")
    print(f"Meta saved : {CBIR_META_PATH}")
    print(f"Vectors    : {embeddings.shape[0]}")
    print(f"Dim        : {embeddings.shape[1]}")


save_cbir_method3_index()

benjamin: indexed 181/181 gallery images
chern_tak: indexed 216/216 gallery images
chillien: indexed 158/158 gallery images
daniel: indexed 194/194 gallery images
dylan: indexed 148/148 gallery images
han_soon: indexed 138/138 gallery images
harry: indexed 72/72 gallery images
isaac: indexed 40/40 gallery images
jing_ang: indexed 197/197 gallery images
jun_wei: indexed 194/194 gallery images
kang_kai: indexed 216/216 gallery images
marion: indexed 202/202 gallery images
ms_nurul: indexed 79/79 gallery images
qi_xuan: indexed 216/216 gallery images
shuang_quan: indexed 193/193 gallery images
wee_xuan: indexed 173/173 gallery images
xiang_yue: indexed 216/216 gallery images
xu_sheng: indexed 209/209 gallery images
yoke_hong: indexed 214/214 gallery images
yong_kang: indexed 170/170 gallery images
zi_herng: indexed 72/72 gallery images
Total train images read: 3498, indexed: 3498

CBIR Method 3 pipeline complete
Index saved: c:\Users\jaft9\School\AttSystem\index\cbir_method3_index.npz
Met